<a href="https://www.kaggle.com/code/simarbirsinghsandhu/realmlp-encoding-fe?scriptVersionId=318054155" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏎️ F1 Pit Stop Prediction | RealMLP Neural Network
### Playground Series S6E5

**⚡ RealMLP_TD via pytabkit | 5-Fold CV | ROC-AUC optimized**

- Trains directly on `1-auc_ovr` metric — no proxy loss
- 24 internal ensemble members
- Original dataset incorporated inside CV folds
- Target encoding on key categorical combinations

> Upvote if useful 🙌 | [XGBoost Baseline](#) | [Full EDA](#)

## 1. Setup

In [1]:
%%time
!pip install -q pytabkit

import random
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder, LabelEncoder
import torch
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.1 MB/s eta 0:00:00:00:01
PyTorch : 2.10.0+cu128
GPU     : Tesla T4
CPU times: user 11.3 s, sys: 3.01 s, total: 14.3 s
Wall time: 42.6 s


## 2. Config

In [2]:
class CFG:
    FOLDS  = 5
    SEED   = 42
    TARGET = 'PitNextLap'
    ID     = 'id'

PARAMS = {
    'random_state'   : 42,
    'verbosity'      : 2,
    'val_metric_name': '1-auc_ovr',   # train directly on AUC

    # ── Architecture ──
    'n_ens'          : 24,
    'hidden_sizes'   : [512, 256, 128],
    'act'            : 'silu',
    'embedding_size' : 6,
    'max_one_hot_cat_size': 18,

    # ── Training ──
    'n_epochs'       : 6,
    'batch_size'     : 256,
    'lr'             : 0.03,
    'wd'             : 0.018,
    'sq_mom'         : 0.98,
    'lr_sched'       : 'lin_cos_log_15',
    'first_layer_lr_factor': 0.25,

    # ── Regularization ──
    'p_drop'         : 0.05,
    'p_drop_sched'   : 'expm4t',
    'ls_eps'         : 0.01,
    'ls_eps_sched'   : 'sqrt_cos',

    # ── PBLD embeddings ──
    'plr_hidden_1'   : 16,
    'plr_hidden_2'   : 8,
    'plr_act_name'   : 'gelu',
    'plr_lr_factor'  : 0.1151,
    'plr_sigma'      : 2.33,

    # ── Preprocessing ──
    'add_front_scale': False,
    'bias_init_mode' : 'neg-uniform-dynamic-2',
    'tfms'           : ['one_hot', 'median_center', 'robust_scale',
                        'smooth_clip', 'embedding', 'l2_normalize'],

    # ── Early stopping ──
    'use_early_stopping'                    : False,
    'early_stopping_additive_patience'      : 10,
    'early_stopping_multiplicative_patience': 1,
}

## 3. Load Data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv')

orig.drop(columns=['Normalized_TyreLife'], inplace=True, errors='ignore')

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"Target rate train : {train[CFG.TARGET].mean():.4f}")
print(f"Target rate orig  : {orig[CFG.TARGET].mean():.4f}")

Train : (439140, 16)
Test  : (188165, 15)
Orig  : (101371, 15)
Target rate train : 0.1990
Target rate orig  : 0.2548


## 4. Feature Engineering

In [4]:
COMPOUND_STINT_MEDIANS = {
    'SOFT':14.0,'MEDIUM':17.0,'HARD':23.0,'INTERMEDIATE':16.0,'WET':14.0
}
COMPOUND_HARDNESS = {
    'SOFT':1,'MEDIUM':2,'HARD':3,'INTERMEDIATE':1,'WET':0
}

def engineer_features(df):
    df = df.copy()
    # ── Core tyre features ───────────────────────────────────
    df['ExpectedStint']       = df['Compound'].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    df['TyreLife_Normalized'] = df['TyreLife'] / df['ExpectedStint']
    df['TyreLife_sq']         = df['TyreLife'] ** 2
    df['TyreLife_sqrt']       = np.sqrt(df['TyreLife'])
    df['TyreLife_log1p']      = np.log1p(df['TyreLife'])
    df['Compound_Hardness']   = df['Compound'].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df['TyreLife_x_Hardness'] = df['TyreLife'] * df['Compound_Hardness']
    df['Norm_x_Hardness']     = df['TyreLife_Normalized'] * df['Compound_Hardness']

    # ── Tyre age flags ────────────────────────────────────────
    df['Is_Fresh']      = (df['TyreLife'] <= 3).astype(np.int8)
    df['Is_Old']        = (df['TyreLife'] > 20).astype(np.int8)
    df['Is_VeryOld']    = (df['TyreLife'] > 40).astype(np.int8)
    df['Is_FirstStint'] = (df['Stint'] == 1).astype(np.int8)

    # ── Degradation ───────────────────────────────────────────
    df['DegRate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)

    # ── Race progress flags ───────────────────────────────────
    df['Early_Race']   = (df['RaceProgress'] < 0.25).astype(np.int8)
    df['Late_Race']    = (df['RaceProgress'] >= 0.75).astype(np.int8)
    df['VeryLate_Race']= (df['RaceProgress'] >= 0.90).astype(np.int8)

    # ── Year flags ────────────────────────────────────────────
    df['Is_2023'] = (df['Year'] == 2023).astype(np.int8)
    df['Is_2025'] = (df['Year'] == 2025).astype(np.int8)

    # ── Ratio features ────────────────────────────────────────
    df['TyreLife_per_Lap']    = df['TyreLife'] / df['LapNumber'].clip(lower=1)
    df['Lap_x_RaceProgress']  = df['LapNumber'] * df['RaceProgress']
    df['Stint_x_TyreLife']    = df['Stint'] * df['TyreLife']
    df['Stint_x_Normalized']  = df['Stint'] * df['TyreLife_Normalized']

    # ── LapTime signal ────────────────────────────────────────
    df['LapDelta_sq']   = df['LapTime_Delta'] ** 2
    df['LapDelta_abs']  = df['LapTime_Delta'].abs()

    return df

# Apply feature engineering
train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

# Separate target
y_orig = orig[CFG.TARGET].copy()
orig   = orig.drop(columns=[CFG.TARGET])

y        = train[CFG.TARGET].copy()
train_id = train[CFG.ID].copy()
test_id  = test[CFG.ID].copy()

X      = train.drop(columns=[CFG.ID, CFG.TARGET, 'ExpectedStint'])
X_test = test.drop(columns=[CFG.ID, 'ExpectedStint'])
orig   = orig.drop(columns=['ExpectedStint'], errors='ignore')

# Align orig columns to X
orig = orig.reindex(columns=X.columns, fill_value=0)

print(f"X      : {X.shape}")
print(f"X_test : {X_test.shape}")
print(f"orig   : {orig.shape}")

X      : (439140, 37)
X_test : (188165, 37)
orig   : (101371, 37)


## 5. Encoding
Key insight from 1st place solution:
* treating floored numericals as categories gives RealMLP
* a different view of the same data — adds diversity

In [5]:
category_map   = {}
IMPORTANT_COMBOS = [('Race','Compound'), ('Race','Year'), ('Driver','Compound')]

NUM_COLS_BASE = ['LapNumber','Stint','TyreLife','Position',
                 'LapTime (s)','RaceProgress','Year','PitStop']

def feature_engineering_nn(df, fit=False):
    df = df.copy()

    # ── Arithmetic interactions ───────────────────────────────
    df['_TyreLife_div_LapNumber'] = (
        df['TyreLife'] / df['LapNumber'].clip(lower=1)
    ).astype('float32')
    df['_LapNumber_div_RaceProgress'] = (
        df['LapNumber'] / (df['RaceProgress'] + 1e-6)
    ).astype('float32')

    extra_num = ['_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress']

    # ── Floor numericals → categorical ───────────────────────
    for col in NUM_COLS_BASE + extra_num:
        cat_name = f"{col}_cat_"
        if col in df.columns:
            if fit:
                codes, uniques = np.floor(df[col]).factorize()
                category_map[col] = uniques
            else:
                uniques  = category_map.get(col, np.array([]))
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes    = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cat_name] = codes.astype(str)

    # ── Count encoding ────────────────────────────────────────
    for col in ['Driver','Compound','Race','Year']:
        count_name = f"_{col}_count"
        if col in df.columns:
            if fit:
                count_map = df[col].astype(str).value_counts()
                category_map[count_name] = count_map
            else:
                count_map = category_map.get(count_name, pd.Series(dtype=int))
            df[count_name] = df[col].astype(str).map(count_map).fillna(0).astype('int32')

    # ── RaceProgress binning ──────────────────────────────────
    bin_name = 'RaceProgress_200_quantile_bin_'
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal',
                              strategy='quantile', subsample=None)
        binned = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32')
        category_map[bin_name] = kb
    else:
        kb     = category_map.get(bin_name)
        binned = kb.transform(df[['RaceProgress']]).ravel().astype('int32') if kb else np.zeros(len(df), dtype='int32')
    df[bin_name] = binned.astype(str)

    # ── Combo categorical features ────────────────────────────
    combo_names = []
    for cols in IMPORTANT_COMBOS:
        combo_name = '_'.join(cols) + '_combo_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques  = category_map.get(combo_name, np.array([]))
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes    = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes.astype(str)

    return df, combo_names

X,      combo_names = feature_engineering_nn(X,      fit=True)
X_test, _           = feature_engineering_nn(X_test, fit=False)
orig,   _           = feature_engineering_nn(orig,   fit=False)

# Align orig again after new features
orig = orig.reindex(columns=X.columns, fill_value=0)

print(f"X      after NN FE : {X.shape}")
print(f"X_test after NN FE : {X_test.shape}")
print(f"combo features     : {combo_names}")

X      after NN FE : (439140, 57)
X_test after NN FE : (188165, 57)
combo features     : ['Race_Compound_combo_', 'Race_Year_combo_', 'Driver_Compound_combo_']


## 6. 5-Fold Training

In [6]:
%%time
skf        = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*50}")
    print(f"  FOLD {fold}/{CFG.FOLDS}")
    print(f"{'='*50}")

    X_tr  = X.iloc[tr_idx].copy()
    y_tr  = y.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Add original dataset to training fold
    X_tr = pd.concat([X_tr, orig.reset_index(drop=True)],
                     axis=0).reset_index(drop=True)
    y_tr = pd.concat([y_tr, y_orig.reset_index(drop=True)],
                     axis=0).reset_index(drop=True)

    # ── Target encoding on combo features inside fold ─────────
    te_cols  = combo_names
    te       = TargetEncoder(cv=CFG.FOLDS, smooth='auto',
                             shuffle=True, random_state=CFG.SEED)
    tr_enc   = te.fit_transform(X_tr[te_cols],  y_tr)
    val_enc  = te.transform(X_val[te_cols])
    tst_enc  = te.transform(X_tst[te_cols])

    te_names = [f"_{c}_TE" for c in te_cols]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    if fold == 1:
        print(f"Total features : {X_tr.shape[1]}")
        print(f"Train rows     : {len(X_tr):,}  (synthetic + original)")

    # ── Train RealMLP ─────────────────────────────────────────
    model = RealMLP_TD_Classifier(**PARAMS)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds       = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds        += fold_test_preds / CFG.FOLDS

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"\nFold {fold} AUC: {fold_auc:.6f}")
    torch.cuda.empty_cache()


  FOLD 1/5
Total features : 60
Train rows     : 452,683  (synthetic + original)
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTim

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/6: val 1-auc_ovr = 0.053476
Epoch 2/6: val 1-auc_ovr = 0.048618
Epoch 3/6: val 1-auc_ovr = 0.046607
Epoch 4/6: val 1-auc_ovr = 0.047132
Epoch 5/6: val 1-auc_ovr = 0.045972
Epoch 6/6: val 1-auc_ovr = 0.045933


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Fold 1 AUC: 0.954067

  FOLD 2/5
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 

...


Fold 4 AUC: 0.952035

  FOLD 5/5
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/6: val 1-auc_ovr = 0.053739
Epoch 2/6: val 1-auc_ovr = 0.049018
Epoch 3/6: val 1-auc_ovr = 0.046780
Epoch 4/6: val 1-auc_ovr = 0.047345
Epoch 5/6: val 1-auc_ovr = 0.046231
Epoch 6/6: val 1-auc_ovr = 0.046222


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Fold 5 AUC: 0.953778
CPU times: user 14min 56s, sys: 1min 58s, total: 16min 54s
Wall time: 16min 36s


## 7. Results

In [7]:
oof_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*40}")
print(f"  OOF ROC-AUC : {oof_auc:.6f}")
print(f"{'='*40}")


  OOF ROC-AUC : 0.952866


## 8. Save OOF + Submission

In [8]:
oof_df = pd.DataFrame({
    CFG.ID    : train_id.values,
    CFG.TARGET: oof_preds,
})
oof_df.to_csv('oof_preds.csv', index=False)
print(f"OOF saved: oof_preds.csv  ({len(oof_df):,} rows)")

sub = pd.DataFrame({
    CFG.ID    : test_id.values,
    CFG.TARGET: test_preds,
})
sub.to_csv('submission.csv', index=False)
print(f"Submission saved: submission.csv")
print(sub.head())

OOF saved: oof_preds.csv  (439,140 rows)
Submission saved: submission.csv
       id  PitNextLap
0  439140    0.005606
1  439141    0.017707
2  439142    0.005802
3  439143    0.271546
4  439144    0.782475


## Summary

| | Score |
|---|---|
| OOF ROC-AUC | 0.952866 |
| Public LB | 0.95253 |

**What makes this strong:**
- Trains directly on AUC (`1-auc_ovr`) not a proxy loss
- 24 internal ensemble members — 3× more than typical
- Treats floored numericals as categorical variables — different view of the data vs tree models
- Original dataset added inside each CV fold

**Why RealMLP complements tree models:**
Neural networks capture different interaction patterns than GBDTs. Blending XGBoost + CatBoost + RealMLP OOFs will push LB higher than any single model.

---
*Upvote if useful 🙌 | [XGBoost Baseline](#) | [Full EDA](#)*